# AdaPert 资源预览

论文: https://arxiv.org/abs/2602.18885v2

展示三类资源的最小查看示例：扰动表达、知识图邻居、语义嵌入维度、ID对应关系。

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

base = Path('..')
print(f'项目目录: {base.resolve()}')

# 加载数据清单
datasets = pd.read_csv(base / 'data_manifest' / 'datasets.csv')
print(f'\n=== 资源清单 ({len(datasets)} 条) ===')
print(datasets[['dataset_id','category','name','download_status']].to_string(index=False))

## 1. 扰动表达数据预览

In [ ]:
h5ad_files = list((base / 'data' / 'raw').glob('**/*.h5ad')) + list((base / 'data' / 'processed').glob('**/*.h5ad'))
if h5ad_files:
    import anndata
    adata = anndata.read_h5ad(h5ad_files[0], backed='r')
    print(f'文件: {h5ad_files[0].name}')
    print(f'形状: {adata.shape} (细胞 × 基因)')
    print(f'obs列: {list(adata.obs.columns)}')
    # 少量对照和处理组细胞
    if 'condition' in adata.obs.columns:
        ctrl = adata[adata.obs['condition'] == 'control'][:5]
        pert = adata[adata.obs['condition'] == 'perturbed'][:5]
        print(f'\n对照细胞(前5): {list(ctrl.obs_names)}')
        print(f'处理细胞(前5): {list(pert.obs_names)}')
    # 一个扰动的标签和细胞数
    if 'perturbation' in adata.obs.columns:
        pert_counts = adata.obs['perturbation'].value_counts()
        print(f'\n扰动数: {pert_counts.shape[0]}')
        print(f'前5个扰动:')
        for p, c in pert_counts.head(5).items():
            print(f'  {p}: {c} 细胞')
else:
    print('未找到h5ad文件。请先下载数据。')
    print('K562/RPE1: https://gwps.wi.mit.edu/')
    print('JURKAT/HEPG2: 原始编号待确认')

## 2. 知识图邻居预览

In [ ]:
string_files = list((base / 'data' / 'raw' / 'string').glob('*.tsv')) + list((base / 'data' / 'raw' / 'string').glob('*.txt'))
if string_files:
    df = pd.read_csv(string_files[0], sep='\t', comment='#', header=None)
    print(f'STRING边文件: {string_files[0].name}')
    print(f'边数: {len(df)}')
    # 一个基因的邻居
    gene = df.iloc[0, 0]
    neighbors = df[df.iloc[:, 0] == gene].iloc[:, 1].tolist()
    print(f'\n基因 {gene} 的邻居(前10): {neighbors[:10]}')
    print(f'邻居数: {len(neighbors)}')
else:
    print('未找到STRING边文件。请从 https://string-db.org/ 下载 v11.5 人类数据。')

## 3. 语义嵌入维度预览

In [ ]:
genept_files = list((base / 'data' / 'raw' / 'genept').glob('*.npy')) + list((base / 'data' / 'raw' / 'genept').glob('*.h5'))
if genept_files:
    f = genept_files[0]
    print(f'GenePT文件: {f.name}')
    if f.suffix == '.npy':
        arr = np.load(f, allow_pickle=True)
        if arr.dtype == object:
            d = arr.item()
            genes = list(d.keys())
            print(f'基因数: {len(genes)}')
            print(f'嵌入维度: {d[genes[0]].shape}')
            print(f'前5个基因: {genes[:5]}')
            print(f'\n基因 {genes[0]} 的嵌入前10维: {d[genes[0]][:10]}')
        else:
            print(f'数组形状: {arr.shape}')
else:
    print('未找到GenePT嵌入文件。请从 https://zenodo.org/records/10833191 下载。')

## 4. 三类资源ID对应关系

In [ ]:
print('=== ID对应关系检查 ===')
print('运行 scripts/check_id_alignment.py 可获取完整统计。')
print()
print('需要核对:')
print('  1. 表达矩阵基因列表')
print('  2. STRING节点（蛋白ID→基因ID映射）')
print('  3. GenePT嵌入键')
print()
print('不静默丢弃无法匹配的基因。')
print('说明哪些基因用于表达输出，哪些只作为图节点。')

## 复现状态提醒

In [ ]:
print('=== 复现状态 ===')
print('当前能做到: 数据查看 + 方法近似实现')
print('严格论文复现待: 官方代码/权重公开')
print()
print('待核对项:')
print('  - AdaPert官方代码、权重及配置入口（不猜测仓库地址）')
print('  - 节点筛选、图规模和部分指标定义存在说明差异')
print('  - JURKAT/HEPG2原始编号未核实')
print('  - 训练/测试官方划分需确认')